In [1]:
import ee
import pandas as pd
import numpy as np

ee.Initialize(project='global-downscaling')

print("Earth Engine initialized.")

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Earth Engine initialized.


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

SAMPLE_INDEX_PATH = Path(
    "Data/Global/pipeline_stage1/station_date_candidates.parquet"
)

STATION_PATH = Path(
    "Data/Global/pipeline_stage1/stations.parquet"
)

# Requested period is derived from the actual sample table,
# but these are useful sanity boundaries.
MIN_EXPECTED_DATE = pd.Timestamp("2015-04-01")
MAX_EXPECTED_DATE = pd.Timestamp("2026-10-03")


print("Configuration ready.")

samples = pd.read_parquet(SAMPLE_INDEX_PATH)

if "stations" not in globals():
    stations = pd.read_parquet(STATION_PATH)

samples["sample_date"] = pd.to_datetime(samples["sample_date"]).dt.normalize()

print(f"Samples:  {len(samples):,}")
print(f"Stations: {stations['station_id'].nunique():,}")

display(samples.head())
display(stations.head())
print("Sample date range:")
print("  min:", samples["sample_date"].min())
print("  max:", samples["sample_date"].max())

print("\nUnique sample dates:", samples["sample_date"].nunique())
print("Unique stations:", samples["station_id"].nunique())

assert samples["sample_date"].min() >= MIN_EXPECTED_DATE
assert samples["sample_date"].max() <= MAX_EXPECTED_DATE

Configuration ready.
Samples:  3,514,643
Stations: 2,210


,sample_id,station_id,sample_date,network,network_name,station,latitude,longitude,elevation_m,climate_zone,source_row_count,min_depth_from_m,max_depth_to_m,sensors,depth_classes
0,0,AMMA-CATCH__Banizoumbou,2015-04-01,AMMA-CATCH,AMMA-CATCH,Banizoumbou,13.5325,2.6604,208.0,BSh,2,0.05,0.05,Campbell-CS616,near_surface
1,1,AMMA-CATCH__Banizoumbou,2015-04-02,AMMA-CATCH,AMMA-CATCH,Banizoumbou,13.5325,2.6604,208.0,BSh,2,0.05,0.05,Campbell-CS616,near_surface
2,2,AMMA-CATCH__Banizoumbou,2015-04-03,AMMA-CATCH,AMMA-CATCH,Banizoumbou,13.5325,2.6604,208.0,BSh,2,0.05,0.05,Campbell-CS616,near_surface
3,3,AMMA-CATCH__Banizoumbou,2015-04-04,AMMA-CATCH,AMMA-CATCH,Banizoumbou,13.5325,2.6604,208.0,BSh,2,0.05,0.05,Campbell-CS616,near_surface
4,4,AMMA-CATCH__Banizoumbou,2015-04-05,AMMA-CATCH,AMMA-CATCH,Banizoumbou,13.5325,2.6604,208.0,BSh,2,0.05,0.05,Campbell-CS616,near_surface


,station_index,station_id,network,network_name,station,latitude,longitude,elevation_m,climate_zone
0,0,AMMA-CATCH__Banizoumbou,AMMA-CATCH,AMMA-CATCH,Banizoumbou,13.53250,2.66040,208.0,BSh
1,1,AMMA-CATCH__Belefoungou-Mid,AMMA-CATCH,AMMA-CATCH,Belefoungou-Mid,9.79506,1.70994,414.0,Aw
2,2,AMMA-CATCH__Belefoungou-Top,AMMA-CATCH,AMMA-CATCH,Belefoungou-Top,9.78986,1.70994,414.0,Aw
3,3,AMMA-CATCH__Nalohou-Mid,AMMA-CATCH,AMMA-CATCH,Nalohou-Mid,9.74530,1.60530,441.0,Aw
4,4,AMMA-CATCH__Nalohou-Top,AMMA-CATCH,AMMA-CATCH,Nalohou-Top,9.74407,1.60580,445.0,Aw


Sample date range:
  min: 2015-04-01 00:00:00
  max: 2026-10-03 00:00:00

Unique sample dates: 4204
Unique stations: 2210


In [3]:
# samples should already be loaded from:
# Data/Global/global_ismn_daily.parquet

station_cols = [
    "network_name",
    "station_id",
    "latitude",
    "longitude",
]

stations_raw = (
    samples[station_cols]
    .dropna(subset=["network_name", "station_id", "latitude", "longitude"])
    .copy()
)

# Force stable string identifiers
stations_raw["network_name"] = stations_raw["network_name"].astype(str)
stations_raw["station_id"] = stations_raw["station_id"].astype(str)

stations_raw["station_key"] = (
    stations_raw["network_name"] + "::" + stations_raw["station_id"]
)

print("Raw station records:", len(stations_raw))
print("Unique station keys:", stations_raw["station_key"].nunique())

Raw station records: 3514643
Unique station keys: 2210


In [4]:
coord_check = (
    stations_raw
    .groupby("station_key")
    .agg(
        lat_min=("latitude", "min"),
        lat_max=("latitude", "max"),
        lon_min=("longitude", "min"),
        lon_max=("longitude", "max"),
        n_lat=("latitude", "nunique"),
        n_lon=("longitude", "nunique"),
    )
)

coord_check["lat_range"] = coord_check["lat_max"] - coord_check["lat_min"]
coord_check["lon_range"] = coord_check["lon_max"] - coord_check["lon_min"]

print("Stations with >1 latitude:", (coord_check["n_lat"] > 1).sum())
print("Stations with >1 longitude:", (coord_check["n_lon"] > 1).sum())

print("\nLargest coordinate ranges:")
print(
    coord_check
    .sort_values(["lat_range", "lon_range"], ascending=False)
    .head(10)
)

Stations with >1 latitude: 40
Stations with >1 longitude: 41

Largest coordinate ranges:
                                   lat_min   lat_max    lon_min    lon_max  \
station_key                                                                  
RISMA::RISMA__MB7                 49.66552  49.66552  -98.00762  -98.00762   
RISMA::RISMA__SK1                 51.33484  51.33484 -106.56494 -106.56494   
RISMA::RISMA__MB9                 49.69462  49.69462  -98.02397  -98.02397   
COSMOS-UK::COSMOS-UK__Heytesbury  51.20289  51.20289   -2.07960   -2.07960   
RISMA::RISMA__MB13                49.93262  49.93262  -99.38716  -99.38716   
RISMA::RISMA__MB6                 49.67877  49.67877  -97.95957  -97.95957   
RISMA::RISMA__MB3                 49.51951  49.51951  -97.95649  -97.95649   
RISMA::RISMA__SK4                 51.38164  51.38164 -106.41583 -106.41583   
ARM::ARM__Medford                 36.81900  36.81900  -97.81990  -97.81990   
RISMA::RISMA__MB17                49.47659  49.47659 

In [5]:
stations = (
    stations_raw
    .groupby(["station_key", "network_name", "station_id"], as_index=False)
    .agg(
        latitude=("latitude", "median"),
        longitude=("longitude", "median"),
    )
)

print("Final unique stations:", len(stations))

Final unique stations: 2210


In [6]:
station_records = stations[
    [
        "station_key",
        "station_id",
        "network_name",
        "latitude",
        "longitude",
    ]
].to_dict("records")

station_features = []

for row in station_records:
    point = ee.Geometry.Point([
        float(row["longitude"]),
        float(row["latitude"]),
    ])

    feature = ee.Feature(
        point,
        {
            "station_key": row["station_key"],
            "station_id": row["station_id"],
            "network": row["network_name"],
        },
    )

    station_features.append(feature)

stations_fc = ee.FeatureCollection(station_features)

print("Earth Engine station count:")
print(stations_fc.size().getInfo())

Earth Engine station count:
2210


In [7]:
clay_img = ee.Image(
    "projects/soilgrids-isric/clay_mean"
)

sand_img = ee.Image(
    "projects/soilgrids-isric/sand_mean"
)

silt_img = ee.Image(
    "projects/soilgrids-isric/silt_mean"
)

print("Clay bands:")
print(clay_img.bandNames().getInfo())

print("\nSand bands:")
print(sand_img.bandNames().getInfo())

print("\nSilt bands:")
print(silt_img.bandNames().getInfo())

Clay bands:
['clay_0-5cm_mean', 'clay_5-15cm_mean', 'clay_15-30cm_mean', 'clay_30-60cm_mean', 'clay_60-100cm_mean', 'clay_100-200cm_mean']

Sand bands:
['sand_0-5cm_mean', 'sand_5-15cm_mean', 'sand_15-30cm_mean', 'sand_30-60cm_mean', 'sand_60-100cm_mean', 'sand_100-200cm_mean']

Silt bands:
['silt_0-5cm_mean', 'silt_5-15cm_mean', 'silt_15-30cm_mean', 'silt_30-60cm_mean', 'silt_60-100cm_mean', 'silt_100-200cm_mean']


In [8]:
clay_0_5 = (
    clay_img
    .select([0])
    .divide(10.0)
    .rename("clay_pct")
)

sand_0_5 = (
    sand_img
    .select([0])
    .divide(10.0)
    .rename("sand_pct")
)

silt_0_5 = (
    silt_img
    .select([0])
    .divide(10.0)
    .rename("silt_pct")
)

soil_image = ee.Image.cat([
    clay_0_5,
    sand_0_5,
    silt_0_5,
])

print(soil_image.bandNames().getInfo())

['clay_pct', 'sand_pct', 'silt_pct']


In [9]:
soil_fc = soil_image.sampleRegions(
    collection=stations_fc,
    properties=[
        "station_key",
        "station_id",
        "network",
    ],
    scale=250,
    geometries=False,
)

print("Soil samples:", soil_fc.size().getInfo())

Soil samples: 2138


In [10]:
dem_collection = ee.ImageCollection(
    "COPERNICUS/DEM/GLO30_2024_1"
)

native_projection = dem_collection.first().projection()

dem = (
    dem_collection
    .mosaic()
    .setDefaultProjection(native_projection)
    .select("DEM")
)

print("DEM loaded.")
print("Projection:")
print(native_projection.getInfo())

DEM loaded.
Projection:
{'type': 'Projection', 'crs': 'EPSG:4326', 'transform': [0.0002777777777777778, 0, 5.9998611111111115, 0, -0.0002777777777777778, 1.000138888888889]}


In [11]:
elevation = dem.rename("elevation_m")

slope = (
    ee.Terrain
    .slope(dem)
    .rename("slope_deg")
)

aspect = (
    ee.Terrain
    .aspect(dem)
    .rename("aspect_deg")
)

terrain_image = ee.Image.cat([
    elevation,
    slope,
    aspect,
])

print(terrain_image.bandNames().getInfo())

['elevation_m', 'slope_deg', 'aspect_deg']


In [12]:
elevation = dem.rename("elevation_m")

slope = (
    ee.Terrain
    .slope(dem)
    .rename("slope_deg")
)

aspect = (
    ee.Terrain
    .aspect(dem)
    .rename("aspect_deg")
)

terrain_image = ee.Image.cat([
    elevation,
    slope,
    aspect,
])

print(terrain_image.bandNames().getInfo())

['elevation_m', 'slope_deg', 'aspect_deg']


In [13]:
terrain_fc = terrain_image.sampleRegions(
    collection=stations_fc,
    properties=[
        "station_key",
        "station_id",
        "network",
    ],
    scale=30,
    geometries=False,
)

print("Terrain samples:", terrain_fc.size().getInfo())

Terrain samples: 2210


In [14]:
soil_export = ee.batch.Export.table.toDrive(
    collection=soil_fc,
    description="global_static_soil",
    folder="GEE_Global_Statics",
    fileNamePrefix="global_static_soil",
    fileFormat="CSV",
    selectors=[
        "station_key",
        "station_id",
        "network",
        "clay_pct",
        "sand_pct",
        "silt_pct",
    ],
)

soil_export.start()

print("Soil export started.")
print(soil_export.status())

Soil export started.
{'state': 'RUNNING', 'description': 'global_static_soil', 'priority': 100, 'creation_timestamp_ms': 1791221216038, 'update_timestamp_ms': 1791221218766, 'start_timestamp_ms': 1791221218644, 'task_type': 'EXPORT_FEATURES', 'attempt': 1, 'id': 'A3QS3LMC7LHGKMRYAYDRVRPL', 'name': 'projects/global-downscaling/operations/A3QS3LMC7LHGKMRYAYDRVRPL'}


In [15]:
terrain_export = ee.batch.Export.table.toDrive(
    collection=terrain_fc,
    description="global_static_terrain",
    folder="GEE_Global_Statics",
    fileNamePrefix="global_static_terrain",
    fileFormat="CSV",
    selectors=[
        "station_key",
        "station_id",
        "network",
        "elevation_m",
        "slope_deg",
        "aspect_deg",
    ],
)

terrain_export.start()

print("Terrain export started.")
print(terrain_export.status())

Terrain export started.
{'state': 'READY', 'description': 'global_static_terrain', 'priority': 100, 'creation_timestamp_ms': 1791221225867, 'update_timestamp_ms': 1791221225867, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'MZGIR65BU75V2QXO5YN7KZLZ', 'name': 'projects/global-downscaling/operations/MZGIR65BU75V2QXO5YN7KZLZ'}


In [17]:
soil_df = pd.read_csv(
    "Data/Global/Statics/global_static_soil.csv"
)

terrain_df = pd.read_csv(
    "Data/Global/Statics/global_static_terrain.csv"
)

print("Soil:")
print(soil_df.shape)
display(soil_df.head())

print("\nTerrain:")
print(terrain_df.shape)
display(terrain_df.head())

Soil:
(2138, 6)


,station_key,station_id,network,clay_pct,sand_pct,silt_pct
0,AMMA-CATCH::AMMA-CATCH__Banizoumbou,AMMA-CATCH__Banizoumbou,AMMA-CATCH,6.9,83.0,10.1
1,AMMA-CATCH::AMMA-CATCH__Belefoungou-Mid,AMMA-CATCH__Belefoungou-Mid,AMMA-CATCH,13.7,68.5,17.7
2,AMMA-CATCH::AMMA-CATCH__Belefoungou-Top,AMMA-CATCH__Belefoungou-Top,AMMA-CATCH,14.7,64.7,20.6
3,AMMA-CATCH::AMMA-CATCH__Nalohou-Mid,AMMA-CATCH__Nalohou-Mid,AMMA-CATCH,15.6,69.0,15.4
4,AMMA-CATCH::AMMA-CATCH__Nalohou-Top,AMMA-CATCH__Nalohou-Top,AMMA-CATCH,15.1,69.5,15.4



Terrain:
(2210, 6)


,station_key,station_id,network,elevation_m,slope_deg,aspect_deg
0,AMMA-CATCH::AMMA-CATCH__Banizoumbou,AMMA-CATCH__Banizoumbou,AMMA-CATCH,208.64137,1.303040,351.192230
1,AMMA-CATCH::AMMA-CATCH__Belefoungou-Mid,AMMA-CATCH__Belefoungou-Mid,AMMA-CATCH,417.68732,3.574303,88.476840
2,AMMA-CATCH::AMMA-CATCH__Belefoungou-Top,AMMA-CATCH__Belefoungou-Top,AMMA-CATCH,422.68190,0.984109,286.395700
3,AMMA-CATCH::AMMA-CATCH__Nalohou-Mid,AMMA-CATCH__Nalohou-Mid,AMMA-CATCH,441.50824,2.156732,353.058700
4,AMMA-CATCH::AMMA-CATCH__Nalohou-Top,AMMA-CATCH__Nalohou-Top,AMMA-CATCH,446.75040,2.320163,18.113632


In [18]:
static_gee = (
    soil_df[
        [
            "station_key",
            "station_id",
            "network",
            "clay_pct",
            "sand_pct",
            "silt_pct",
        ]
    ]
    .merge(
        terrain_df[
            [
                "station_key",
                "elevation_m",
                "slope_deg",
                "aspect_deg",
            ]
        ],
        on="station_key",
        how="outer",
        validate="one_to_one",
    )
)

print(static_gee.shape)
display(static_gee.head())

(2210, 9)


,station_key,station_id,network,clay_pct,sand_pct,silt_pct,elevation_m,slope_deg,aspect_deg
0,AMMA-CATCH::AMMA-CATCH__Banizoumbou,AMMA-CATCH__Banizoumbou,AMMA-CATCH,6.9,83.0,10.1,208.64137,1.303040,351.192230
1,AMMA-CATCH::AMMA-CATCH__Belefoungou-Mid,AMMA-CATCH__Belefoungou-Mid,AMMA-CATCH,13.7,68.5,17.7,417.68732,3.574303,88.476840
2,AMMA-CATCH::AMMA-CATCH__Belefoungou-Top,AMMA-CATCH__Belefoungou-Top,AMMA-CATCH,14.7,64.7,20.6,422.68190,0.984109,286.395700
3,AMMA-CATCH::AMMA-CATCH__Nalohou-Mid,AMMA-CATCH__Nalohou-Mid,AMMA-CATCH,15.6,69.0,15.4,441.50824,2.156732,353.058700
4,AMMA-CATCH::AMMA-CATCH__Nalohou-Top,AMMA-CATCH__Nalohou-Top,AMMA-CATCH,15.1,69.5,15.4,446.75040,2.320163,18.113632


In [19]:
static_cols = [
    "clay_pct",
    "sand_pct",
    "silt_pct",
    "elevation_m",
    "slope_deg",
    "aspect_deg",
]

print("Missing values:")
display(static_gee[static_cols].isna().sum())

print("\nSummary:")
display(static_gee[static_cols].describe())

texture_sum = (
    static_gee["clay_pct"]
    + static_gee["sand_pct"]
    + static_gee["silt_pct"]
)

print(texture_sum.describe())

print(
    "Invalid slope:",
    (~static_gee["slope_deg"].dropna().between(0, 90)).sum()
)

print(
    "Invalid aspect:",
    (~static_gee["aspect_deg"].dropna().between(0, 360)).sum()
)

Missing values:


clay_pct       72
sand_pct       72
silt_pct       72
elevation_m     0
slope_deg       0
aspect_deg      0
dtype: int64


Summary:


,clay_pct,sand_pct,silt_pct,elevation_m,slope_deg,aspect_deg
count,2138.000000,2138.000000,2138.000000,2210.000000,2210.000000,2210.000000
mean,21.306361,42.500281,36.194528,1128.499936,5.040476,178.037165
std,8.661856,17.150351,12.489186,1208.008330,5.567498,102.764359
min,1.900000,1.500000,2.000000,-49.400093,0.000000,-0.000000
25%,15.300000,31.625000,27.400000,178.541732,1.201167,89.535124
50%,21.400000,39.900000,37.300000,518.289780,3.304781,178.180325
75%,26.600000,54.675000,42.900000,1923.530925,6.863095,263.980805
max,61.800000,96.000000,80.500000,4945.571000,46.522507,359.691960


count    2138.000000
mean      100.001169
std         0.052930
min        99.900000
25%       100.000000
50%       100.000000
75%       100.000000
max       100.100000
dtype: float64
Invalid slope: 0
Invalid aspect: 0
